# Suspension Model Test & Debug
서스펜션 모델 테스트 및 디버깅 노트북 (7-DOF 기반)

## 목적
- SuspensionModel 4개 코너 동작 확인
- **입력**: T_susp, X_body (차체 상태 벡터), z_road
  - X_body = [heave, roll, pitch, heave_dot, roll_rate, pitch_rate]
- **출력**: F_s (서스펜션 힘), F_z (타이어 수직력)
- 7-DOF 동역학 검증: m_u × z_u_ddot = F_z - F_s - m_u × g
- 코너별 좌표계 부호 검증 (YAML corner_signs)
- 평형 상태 테스트: 모든 입력 = 0일 때 힘 평형 확인

In [1]:
# 필요한 라이브러리 import
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 프로젝트 루트를 Python path에 추가
project_root = Path.cwd().parent.parent.parent.parent.parent
sys.path.insert(0, str(project_root))

# SuspensionModel import
from vehicle_sim.models.e_corner.suspension.suspension_model import SuspensionModel

print("Import 성공!")

Import 성공!


## 1. 기본 동작 테스트
4개 코너 SuspensionModel 생성 및 파라미터 확인

In [ ]:
# 4개 코너 모델 생성
corners = {}
for corner_id in ['FL', 'FR', 'RL', 'RR']:
    corners[corner_id] = SuspensionModel(corner_id)

print("=== Suspension Model Parameters (FL 기준) ===")
susp = corners['FL']
print(f"스프링 강성 K_spring: {susp.params.K_spring} N/m")
print(f"댐퍼 감쇠 C_damper: {susp.params.C_damper} N*s/m")
print(f"평형 CG 높이 z_CG0: {susp.params.z_CG0} m")
print(f"서스펜션 레스트 길이 L_s0: {susp.params.L_s0} m")
print(f"타이어 반지름 R_w: {susp.params.R_w} m")
print(f"볼 스크류 리드 lead: {susp.params.lead} m/rev")
print(f"기계적 효율 efficiency: {susp.params.efficiency}")
print(f"트랙 폭 L_track: {susp.params.L_track} m")
print(f"휠베이스 L_wheelbase: {susp.params.L_wheelbase} m")

print("\n=== 서스펜션 스트로크 한계 (위치 제한 방식) ===")
print(f"최대 압축 delta_s_min: {susp.params.delta_s_min} m")
print(f"최대 신장 delta_s_max: {susp.params.delta_s_max} m")

print("\n=== 타이어 수직 파라미터 ===")
print(f"타이어 강성 K_t: {susp.tire_params.K_t} N/m")
print(f"타이어 댐핑 C_t: {susp.tire_params.C_t} N*s/m")


print("\n=== 언스프렁 파라미터 ===")
print(f"언스프렁 질량 m_u: {susp.unsprung_params.m_u} kg")
print(f"중력가속도 g: {susp.unsprung_params.g} m/s²")

print("\n=== 코너별 좌표계 부호 (Corner Signs) ===")
for corner_id, model in corners.items():
    print(f"{corner_id}: sign_roll={model.params.sign_roll:+d}, sign_pitch={model.params.sign_pitch:+d}")

print("\n=== 초기 상태 (FL 기준) ===")
state = susp.get_state()
print(f"Suspension 상태:")
print(f"  F_s: {state['F_s']:.2f} N")
print(f"  F_z: {state['F_z']:.2f} N")
print(f"  F_active: {state['F_active']:.2f} N")
print(f"  F_spring: {state['F_spring']:.2f} N")
print(f"  F_damper: {state['F_damper']:.2f} N")

## 2. 차체 자세 → 바퀴 높이 테스트
**입력**: roll, pitch, heave_dev (차체 자세)  
**출력**: z_body_abs (4개 코너별 차체 절대 높이)  
**검증**: 코너별 좌표계 부호 정상 동작

In [3]:
# 테스트 입력: Roll만 적용
roll = 0.1  # rad (약 5.7도)
pitch = 0.0
heave_dev = 0.0

print("=== Roll 운동 테스트 ===")
print(f"입력: roll={roll} rad, pitch={pitch} rad, heave_dev={heave_dev} m\n")

z_body_roll = {}
for corner_id, model in corners.items():
    z_CG = model.params.z_CG0 + heave_dev
    z_body_abs = model._calculate_body_height(z_CG, roll, pitch)
    z_body_roll[corner_id] = z_body_abs
    print(f"{corner_id}: z_body_abs = {z_body_abs:.4f} m")

print("\n검증: Roll 시 좌측 상승(+), 우측 하강(-) 대칭")
print(f"  FL vs FR: {z_body_roll['FL']:.4f} vs {z_body_roll['FR']:.4f}")
print(f"  RL vs RR: {z_body_roll['RL']:.4f} vs {z_body_roll['RR']:.4f}")

=== Roll 운동 테스트 ===
입력: roll=0.1 rad, pitch=0.0 rad, heave_dev=0.0 m

FL: z_body_abs = 1.0470 m
FR: z_body_abs = 0.8870 m
RL: z_body_abs = 1.0470 m
RR: z_body_abs = 0.8870 m

검증: Roll 시 좌측 상승(+), 우측 하강(-) 대칭
  FL vs FR: 1.0470 vs 0.8870
  RL vs RR: 1.0470 vs 0.8870


In [4]:
# 테스트 입력: Pitch만 적용
roll = 0.0
pitch = 0.05  # rad (약 2.9도)
heave_dev = 0.0

print("=== Pitch 운동 테스트 ===")
print(f"입력: roll={roll} rad, pitch={pitch} rad, heave_dev={heave_dev} m\n")

z_body_pitch = {}
for corner_id, model in corners.items():
    z_CG = model.params.z_CG0 + heave_dev
    z_body_abs = model._calculate_body_height(z_CG, roll, pitch)
    z_body_pitch[corner_id] = z_body_abs
    print(f"{corner_id}: z_body_abs = {z_body_abs:.4f} m")

print("\n검증: Pitch 시 전방 상승(+), 후방 하강(-) 대칭")
print(f"  FL vs RL: {z_body_pitch['FL']:.4f} vs {z_body_pitch['RL']:.4f}")
print(f"  FR vs RR: {z_body_pitch['FR']:.4f} vs {z_body_pitch['RR']:.4f}")

=== Pitch 운동 테스트 ===
입력: roll=0.0 rad, pitch=0.05 rad, heave_dev=0.0 m

FL: z_body_abs = 0.8970 m
FR: z_body_abs = 0.8970 m
RL: z_body_abs = 1.0370 m
RR: z_body_abs = 1.0370 m

검증: Pitch 시 전방 상승(+), 후방 하강(-) 대칭
  FL vs RL: 0.8970 vs 1.0370
  FR vs RR: 0.8970 vs 1.0370


In [5]:
# 테스트 입력: Heave만 적용
roll = 0.0
pitch = 0.0
heave_dev = 0.05  # m

print("=== Heave 운동 테스트 ===")
print(f"입력: roll={roll} rad, pitch={pitch} rad, heave_dev={heave_dev} m\n")

z_body_heave = {}
for corner_id, model in corners.items():
    z_CG = model.params.z_CG0 + heave_dev
    z_body_abs = model._calculate_body_height(z_CG, roll, pitch)
    z_body_heave[corner_id] = z_body_abs
    print(f"{corner_id}: z_body_abs = {z_body_abs:.4f} m")

print("\n검증: Heave 시 4개 코너 동일")
expected = model.params.z_CG0 + heave_dev
all_equal = all(abs(z - expected) < 1e-10 for z in z_body_heave.values())
print(f"  모든 코너 동일: {all_equal} (예상값: {expected} m)")

=== Heave 운동 테스트 ===
입력: roll=0.0 rad, pitch=0.0 rad, heave_dev=0.05 m

FL: z_body_abs = 1.0170 m
FR: z_body_abs = 1.0170 m
RL: z_body_abs = 1.0170 m
RR: z_body_abs = 1.0170 m

검증: Heave 시 4개 코너 동일
  모든 코너 동일: True (예상값: 1.017 m)


## 3. 액티브 서스펜션 토크 테스트
**입력**: T_susp (볼 스크류 토크)  
**출력**: F_active (액티브 힘)  
**공식**: F_active = (2π × efficiency / lead) × T_susp

In [6]:
# 테스트: 액티브 토크 → 힘 변환
T_susp_test = 10.0  # N*m

susp = corners['FL']
F_active_calc = susp._calculate_active_force(T_susp_test)

# 수동 검증
F_active_expected = (2 * np.pi * susp.params.efficiency / susp.params.lead) * T_susp_test

print("=== 액티브 서스펜션 토크 테스트 ===")
print(f"입력 토크 T_susp: {T_susp_test} N*m")
print(f"\n파라미터:")
print(f"  lead = {susp.params.lead} m/rev")
print(f"  efficiency = {susp.params.efficiency}")
print(f"  F_active_max = {susp.params.F_active_max} N (포화 한계)")
print(f"\n출력 힘 F_active: {F_active_calc:.2f} N")
print(f"예상 힘 (수동 계산, 포화 전): {F_active_expected:.2f} N")
print(f"포화 여부: {'포화됨 ⚠️' if abs(F_active_calc) >= susp.params.F_active_max else '정상'}")
print(f"\n검증: 모델 출력 = min(예상값, F_active_max)")
print(f"  → {F_active_calc:.2f} = min({F_active_expected:.2f}, {susp.params.F_active_max:.2f})")

=== 액티브 서스펜션 토크 테스트 ===
입력 토크 T_susp: 10.0 N*m

파라미터:
  lead = 0.01 m/rev
  efficiency = 0.9
  F_active_max = 4000.0 N (포화 한계)

출력 힘 F_active: 4000.00 N
예상 힘 (수동 계산, 포화 전): 5654.87 N
포화 여부: 포화됨 ⚠️

검증: 모델 출력 = min(예상값, F_active_max)
  → 4000.00 = min(5654.87, 4000.00)


## 4. 평형 상태 테스트 (Equilibrium Test)
**목적**: 정지 상태에서 서스펜션 힘 평형 확인  
**입력**: 모든 입력 = 0 (roll=0, pitch=0, heave=0, z_road=0)  
**출력**: 4개 코너별 F_sus, F_z_tire, delta_s, delta_t  
**검증**: 정적 평형 상태에서의 힘 관계 확인

In [ ]:
# 단일 코너 평형 테스트 (FL)
dt = 0.01
t_end = 3.0
time_steps = int(t_end / dt)
time = np.arange(0, t_end, dt)

# FL 코너만 사용
model = corners['FL']
model.reset()

# 데이터 저장
data = {
    'time': time,
    'heave': np.zeros(time_steps),
    'z_road': np.zeros(time_steps),
    'z_u_abs': np.zeros(time_steps),
    'z_body_abs': np.zeros(time_steps),
    'delta_s': np.zeros(time_steps),
    'delta_t': np.zeros(time_steps),
    'F_s': np.zeros(time_steps),
    'F_z': np.zeros(time_steps),
    'F_spring': np.zeros(time_steps),
    'F_damper': np.zeros(time_steps),
}

print("=== 평형 상태 테스트 시작 ===")
print(f"초기 z_u_abs: {model.state.z_u_abs:.4f} m")
print(f"z_CG0: {model.params.z_CG0:.4f} m")
print(f"L_s0: {model.params.L_s0:.4f} m")
print(f"R_w: {model.params.R_w:.4f} m\n")

# 시뮬레이션 실행 (모든 입력 = 0)
for i in range(time_steps):
    t = time[i]
    
    # 평형 상태: X_body 모두 0
    X_body = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])  # [heave, roll, pitch, heave_dot, roll_rate, pitch_rate]
    z_road = 0.0
    
    # 업데이트
    F_s, F_z = model.update(
        dt=dt,
        T_susp=0.0,
        X_body=X_body,
        z_road=z_road
    )
    
    # 데이터 저장
    data['heave'][i] = X_body[0]
    data['z_road'][i] = z_road
    data['z_u_abs'][i] = model.state.z_u_abs
    data['z_body_abs'][i] = model.state.z_body_abs
    data['delta_s'][i] = model.state.delta_s
    data['delta_t'][i] = model.state.delta_t
    data['F_s'][i] = F_s
    data['F_z'][i] = F_z
    data['F_spring'][i] = model.state.F_spring
    data['F_damper'][i] = model.state.F_damper
    
    # 초반 5스텝 출력
    if i < 5:
        print(f"[t={t:.3f}s] 평형 상태 (X_body = 0)")
        print(f"  z_body_abs={model.state.z_body_abs:.4f}m, z_u_abs={model.state.z_u_abs:.4f}m")
        print(f"  delta_s={model.state.delta_s*1000:.2f}mm, delta_t={model.state.delta_t*1000:.2f}mm")
        print(f"  F_spring={model.state.F_spring:.1f}N, F_damper={model.state.F_damper:.1f}N")
        print(f"  F_s={F_s:.1f}N, F_z={F_z:.1f}N\n")

print("평형 상태 테스트 완료!")
print(f"\n=== 평형 상태 최종 값 ===")
print(f"F_z: {data['F_z'][-1]:.1f} N (일정)")
print(f"F_s: {data['F_s'][-1]:.1f} N (일정)")
print(f"delta_s: {data['delta_s'][-1]*1000:.2f} mm (일정)")
print(f"delta_t: {data['delta_t'][-1]*1000:.2f} mm (일정)")
print(f"\n=== 평형 조건 검증 ===")
print(f"언스프렁 질량 무게: m_u × g = {model.unsprung_params.m_u * model.unsprung_params.g:.1f} N")
print(f"F_z - F_s = {data['F_z'][-1] - data['F_s'][-1]:.1f} N")
print(f"차이: {abs(data['F_z'][-1] - data['F_s'][-1] - model.unsprung_params.m_u * model.unsprung_params.g):.1f} N (≈ 0이면 평형)")

In [ ]:
# 시각화 - 평형 상태 확인
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

t = data['time']

# 1. 입력 (모두 0)
axes[0, 0].plot(t, data['heave']*1000, 'b-', label='Heave')
axes[0, 0].plot(t, data['z_road']*1000, 'r-', label='Road')
axes[0, 0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[0, 0].set_ylabel('mm')
axes[0, 0].set_title('Input: Heave & Road (All Zero)')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# 2. 절대 높이 (일정)
axes[0, 1].plot(t, data['z_body_abs'], 'b-', label='z_body_abs')
axes[0, 1].plot(t, data['z_u_abs'], 'r-', label='z_u_abs')
axes[0, 1].set_ylabel('m')
axes[0, 1].set_title('Absolute Heights (Constant)')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# 3. 편차 (delta_s) - 일정
axes[1, 0].plot(t, data['delta_s']*1000, 'b-', label='delta_s')
axes[1, 0].axhline(y=model.params.delta_s_max*1000, color='r', linestyle='--', alpha=0.5, label='Limits')
axes[1, 0].axhline(y=model.params.delta_s_min*1000, color='r', linestyle='--', alpha=0.5)
axes[1, 0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1, 0].set_ylabel('mm')
axes[1, 0].set_title('Suspension Stroke (Equilibrium)')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# 4. 편차 (delta_t) - 일정
axes[1, 1].plot(t, data['delta_t']*1000, 'b-', label='delta_t')
axes[1, 1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1, 1].set_ylabel('mm')
axes[1, 1].set_title('Tire Deflection (Equilibrium)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

# 5. 서스펜션 힘 - 일정
axes[2, 0].plot(t, data['F_spring'], 'b-', label='F_spring', alpha=0.7)
axes[2, 0].plot(t, data['F_damper'], 'g-', label='F_damper (=0)', alpha=0.7)
axes[2, 0].plot(t, data['F_s'], 'r-', linewidth=2, label='F_s (total)')
axes[2, 0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[2, 0].set_ylabel('N')
axes[2, 0].set_xlabel('Time [s]')
axes[2, 0].set_title('Suspension Forces (Equilibrium)')
axes[2, 0].grid(True, alpha=0.3)
axes[2, 0].legend()

# 6. 타이어 힘 - 일정
axes[2, 1].plot(t, data['F_z'], 'b-', linewidth=2, label='F_z')
axes[2, 1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[2, 1].set_ylabel('N')
axes[2, 1].set_xlabel('Time [s]')
axes[2, 1].set_title('Tire Vertical Force (Equilibrium)')
axes[2, 1].grid(True, alpha=0.3)
axes[2, 1].legend()

fig.suptitle('FL Corner - Equilibrium Test (All Inputs = 0)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ 평형 상태에서 모든 값이 일정하게 유지됩니다.")
print("✓ F_damper = 0 (속도 변화 없음)")
print("✓ F_z - F_s = m_u × g (힘 평형)")

## 5. 액티브 모터 힘 확인
**목적**: 액티브 토크 입력에 대한 힘 출력 확인  
**공식**: F_active = (2π × efficiency / lead) × T_susp  
**포화 한계**: F_active_max = 4000 N (YAML 설정)  
**검증**: 다양한 토크 값에 대한 F_active 계산 및 포화 확인

In [ ]:
# 액티브 모터 힘 테스트
susp = corners['FL']

# 다양한 토크 값 테스트
T_susp_values = np.array([-10, -5, 0, 5, 10, 20])
F_active_values = np.array([susp._calculate_active_force(T) for T in T_susp_values])

# 파라미터 출력
print("=== 액티브 서스펜션 파라미터 ===")
print(f"볼 스크류 리드 (lead): {susp.params.lead} m/rev")
print(f"기계적 효율 (efficiency): {susp.params.efficiency}")
print(f"포화 한계 (F_active_max): {susp.params.F_active_max} N")
print(f"변환 게인: (2π × η / lead) = {(2*np.pi*susp.params.efficiency/susp.params.lead):.2f} N/(N·m)")

# 결과 출력
print("\n=== 토크 → 힘 변환 결과 ===")
print(f"{'T_susp [N·m]':>15} | {'F_active [N]':>15} | {'상태':>10}")
print("-" * 45)
for T, F in zip(T_susp_values, F_active_values):
    F_expected = (2*np.pi*susp.params.efficiency/susp.params.lead) * T
    status = "포화" if abs(F) >= susp.params.F_active_max else "정상"
    print(f"{T:>15.1f} | {F:>15.1f} | {status:>10}")

# 시각화
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.plot(T_susp_values, F_active_values, 'bo-', linewidth=2, markersize=8, label='F_active (Actual)')

# 포화 없는 이론 곡선
T_theory = np.linspace(-10, 20, 100)
F_theory = (2*np.pi*susp.params.efficiency/susp.params.lead) * T_theory
ax.plot(T_theory, F_theory, 'r--', linewidth=1.5, alpha=0.5, label='Theory (No Saturation)')

# 포화 한계선
ax.axhline(y=susp.params.F_active_max, color='orange', linestyle='--', linewidth=2, alpha=0.7, label=f'F_active_max = {susp.params.F_active_max}N')
ax.axhline(y=-susp.params.F_active_max, color='orange', linestyle='--', linewidth=2, alpha=0.7)

ax.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='k', linestyle='-', alpha=0.3)
ax.set_xlabel('T_susp [N·m]', fontsize=12, fontweight='bold')
ax.set_ylabel('F_active [N]', fontsize=12, fontweight='bold')
ax.set_title('Active Suspension Force vs Motor Torque (with Saturation)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc='lower right')

# 게인 및 포화 정보 표시
gain = (2*np.pi*susp.params.efficiency/susp.params.lead)
info_text = f'Gain = {gain:.2f} N/(N·m)\nlead = {susp.params.lead} m/rev\neta = {susp.params.efficiency}\nF_max = +/-{susp.params.F_active_max} N'
ax.text(0.05, 0.50, info_text, 
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n✓ 액티브 힘은 토크에 선형 비례합니다 (포화 전까지).")
print(f"✓ 양의 토크(+) → 차체를 위로 미는 힘(+)")
print(f"✓ 음의 토크(-) → 차체를 아래로 당기는 힘(-)")
print(f"✓ 포화 한계: ±{susp.params.F_active_max} N")